# The aim is to prepare the beans dataset
https://github.com/earthspecies/beans/tree/main

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

In [42]:
from tqdm.notebook import tqdm

In [25]:
from esp_data import AnyPath

In [45]:
import esp_data.file_io.functional as F

In [26]:
from pathlib import Path

In [ ]:
import duckdb

In [2]:
dataset_names_and_types = {
    "cbi": "classification",
    "watkins": "classification",
    "humbugdb": "classification",
    "egyptian_fruit_bats": "classification",
    "dogs": "classification",
    "hiceas": "detection",
    "dcase": "detection",
    "enabirds": "detection",
    "rfcx": "detection",
    "hainan_gibbons": "detection",
    "esc50": "classification",
    "speech-commands": "classification"
}

In [14]:
paths_to_annotations = {
    "cbi": {"train": "gs://foundation-model-data/audio/cbi/annotations.train.csv",
            "test": "gs://foundation-model-data/audio/cbi/annotations.test.csv",
            "validation": "gs://foundation-model-data/audio/cbi/annotations.valid.csv",
           },
    "watkins": {"train": "gs://foundation-model-data/audio/watkins/annotations.train.csv",
                "test": "gs://foundation-model-data/audio/watkins/annotations.test.csv",
                "validation": "gs://foundation-model-data/audio/watkins/annotations.valid.csv",
               },
    "humbugdb": {
        "train": "gs://foundation-model-data/audio/HumBugDB/data/metadata/train.csv",
        "test": "gs://foundation-model-data/audio/HumBugDB/data/metadata/test.csv",
        "validation": "gs://foundation-model-data/audio/HumBugDB/data/metadata/valid.csv",
    },
    "egyptian_fruit_bats": {
        "train": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.train.csv",
        "test": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.test.csv",
        "validation": "gs://foundation-model-data/audio/egyptian_fruit_bats/annotations.valid.csv",
    },
    "dogs": {
        "train": "gs://foundation-model-data/audio/dogs/annotations.train.csv",
        "test": "gs://foundation-model-data/audio/dogs/annotations.test.csv",
        "validation": "gs://foundation-model-data/audio/dogs/annotations.valid.csv",
    },
    "hiceas": {
        "train": "gs://foundation-model-data/data/hiceas-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/hiceas-detection/test.jsonl",
        "validation": "gs://foundation-model-data/data/hiceas-detection/valid.jsonl",
    },
    "dcase": {
        "train": "gs://foundation-model-data/data/dcase-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/dcase-detection/test.jsonl",
        "validation": "gs://foundation-model-data/data/dcase-detection/valid.jsonl",
    },
    "enabirds": {
        "train": "gs://foundation-model-data/data/enabirds-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/enabirds-detection/test.jsonl",
        "validation": "gs://foundation-model-data/data/enabirds-detection/valid.jsonl",
    },
    "rfcx": {
        "train": "gs://foundation-model-data/data/rfcx-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/rfcx-detection/test.jsonl",
        "validation": "gs://foundation-model-data/data/rfcx-detection/valid.jsonl",
    },
    "hainan_gibbons": {
        "train": "gs://foundation-model-data/data/hainan-gibbons-detection/train.jsonl",
        "test": "gs://foundation-model-data/data/hainan-gibbons-detection/test.jsonl",
        "validation": "gs://foundation-model-data/data/hainan-gibbons-detection/valid.jsonl",
    },
    "esc50-all": {
        "train": "gs://foundation-model-data/data/esc50-all/train.jsonl",
        "test": "gs://foundation-model-data/data/esc50-all/test.jsonl",
        "validation": "gs://foundation-model-data/data/esc50-all/valid.jsonl"
    },
    "speech-commands": {
        "train": "gs://foundation-model-data/data/speech-commands-classification/train.jsonl",
        "test": "gs://foundation-model-data/data/speech-commands-classification/test.jsonl",
        "validation": "gs://foundation-model-data/data/speech-commands-classification/valid.jsonl"
    }
}

In [41]:
beans_root = AnyPath("gs://esp-ml-datasets/beans/v0.1.0/raw/")
beans_root

GSPath('gs://esp-ml-datasets/beans/v0.1.0/raw/')

In [16]:
train_dfs = {}
test_dfs = {}
validation_dfs = {}

for dsname, splits in paths_to_annotations.items():

    for s,v in splits.items():

        if "jsonl" in v:
            df = pd.read_json(v, lines=True, orient="records")
        elif "csv" in v:
            df = pd.read_csv(v)

        if s == "train":
            train_dfs[dsname] = df
        elif s == "test":
            test_dfs[dsname] = df
        else:
            validation_dfs[dsname] = df

In [17]:
train_cols = [list(v.columns) for k,v in train_dfs.items()]

In [21]:
paths_to_annotations.keys()

dict_keys(['cbi', 'watkins', 'humbugdb', 'egyptian_fruit_bats', 'dogs', 'hiceas', 'dcase', 'enabirds', 'rfcx', 'hainan_gibbons', 'esc50-all', 'speech-commands'])

In [18]:
train_cols

[['Unnamed: 0', 'path', 'label', 'split'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['Unnamed: 0', 'path', 'label'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'answer'],
 ['instruction', 'path', 'label'],
 ['instruction', 'path', 'label']]

# Fix all

## CBI

In [20]:
train_dfs["cbi"].head(5)

,Unnamed: 0,path,label,split
0,1,data/cbi/wav/XC135454.wav,aldfly,train
1,2,data/cbi/wav/XC135455.wav,aldfly,train
2,3,data/cbi/wav/XC135456.wav,aldfly,train
3,4,data/cbi/wav/XC135457.wav,aldfly,train
4,5,data/cbi/wav/XC135459.wav,aldfly,train


### create file_name and local_path cols

In [29]:
train_dfs["cbi"]["file_name"] = train_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
train_dfs["cbi"]["local_path"] = train_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)

In [36]:
test_dfs["cbi"]["file_name"] = test_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
test_dfs["cbi"]["local_path"] = test_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)
validation_dfs["cbi"]["file_name"] = validation_dfs["cbi"]["path"].apply(lambda x: Path(x).name)
validation_dfs["cbi"]["local_path"] = validation_dfs["cbi"]["file_name"].apply(lambda x: "audio/cbi/wav/" + x)

### drop split column and Unnamed:0

In [ ]:
train_dfs["cbi"] = train_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)

In [37]:
test_dfs["cbi"] = test_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)
validation_dfs["cbi"] = validation_dfs["cbi"].drop(["Unnamed: 0", "path", "split"], axis=1)

In [31]:
train_dfs["cbi"].columns

Index(['label', 'file_name', 'local_path'], dtype='object')

In [38]:
print(test_dfs["cbi"], validation_dfs["cbi"])

       label     file_name                  local_path
0     aldfly  XC137570.wav  audio/cbi/wav/XC137570.wav
1     aldfly  XC154310.wav  audio/cbi/wav/XC154310.wav
2     aldfly  XC157462.wav  audio/cbi/wav/XC157462.wav
3     aldfly  XC180091.wav  audio/cbi/wav/XC180091.wav
4     aldfly  XC182735.wav  audio/cbi/wav/XC182735.wav
...      ...           ...                         ...
3615  yetvir  XC376747.wav  audio/cbi/wav/XC376747.wav
3616  yetvir  XC417474.wav  audio/cbi/wav/XC417474.wav
3617  yetvir  XC421946.wav  audio/cbi/wav/XC421946.wav
3618  yetvir  XC467630.wav  audio/cbi/wav/XC467630.wav
3619  yetvir  XC471316.wav  audio/cbi/wav/XC471316.wav

[3620 rows x 3 columns]        label     file_name                  local_path
0     aldfly  XC134874.wav  audio/cbi/wav/XC134874.wav
1     aldfly  XC135883.wav  audio/cbi/wav/XC135883.wav
2     aldfly  XC138639.wav  audio/cbi/wav/XC138639.wav
3     aldfly  XC139577.wav  audio/cbi/wav/XC139577.wav
4     aldfly  XC154449.wav  audio/cbi/wa

### Perform a check that every audio file exists

In [55]:
dsname = "cbi"

def check_audio_exists(dsname, extensions: list[str] = ["wav", "flac", "mp3"]):
    audio_root = beans_root / f"audio/{dsname}" 
    all_audios = []
    for e in extensions:
        all_audios.extend(F.list_files(audio_root, f"**/*.{e}", use_fs=True))
    print(f"Found {len(all_audios)} audio files for {dsname}")
    
    for split, df in zip(["train", "test", "val"], [train_dfs[dsname], test_dfs[dsname], validation_dfs[dsname]]):
        files_in_wavs = df["local_path"].apply(lambda x: str(beans_root / x)).isin(all_audios).sum()
        assert files_in_wavs == len(df), f"Woops! Mismatch for {split} in {dsname}"

## Watkins

In [52]:
train_dfs["watkins"].sample(5)

,label,file_name,local_path
492,Atlantic_Spotted_Dolphin,6102500H.wav,audio/watkins/6102500H.wav
663,Pantropical_Spotted_Dolphin,9400801H.wav,audio/watkins/9400801H.wav
117,Spinner_Dolphin,71010018.wav,audio/watkins/71010018.wav
2,Clymene_Dolphin,83035002.wav,audio/watkins/83035002.wav
175,Minke_Whale,64104003.wav,audio/watkins/64104003.wav


In [33]:
train_dfs["watkins"]["file_name"] = train_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
train_dfs["watkins"]["local_path"] = train_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

In [39]:
test_dfs["watkins"]["file_name"] = test_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
test_dfs["watkins"]["local_path"] = test_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

validation_dfs["watkins"]["file_name"] = validation_dfs["watkins"]["path"].apply(lambda x: Path(x).name)
validation_dfs["watkins"]["local_path"] = validation_dfs["watkins"]["file_name"].apply(lambda x: "audio/watkins/" + x)

In [ ]:
train_dfs["watkins"] = train_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)

In [53]:

test_dfs["watkins"] = test_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)
validation_dfs["watkins"] = validation_dfs["watkins"].drop(["Unnamed: 0", "path"], axis=1)
test_dfs["watkins"].columns

Index(['label', 'file_name', 'local_path'], dtype='object')

In [54]:
validation_dfs["watkins"].head(5)

,label,file_name,local_path
0,Clymene_Dolphin,8300604P.wav,audio/watkins/8300604P.wav
1,Clymene_Dolphin,83006031.wav,audio/watkins/83006031.wav
2,Clymene_Dolphin,83006022.wav,audio/watkins/83006022.wav
3,Clymene_Dolphin,8303503O.wav,audio/watkins/8303503O.wav
4,Clymene_Dolphin,8300600B.wav,audio/watkins/8300600B.wav


In [56]:
train_dfs["watkins"]["label"].unique()

array(['Clymene_Dolphin', 'Bottlenose_Dolphin', 'Spinner_Dolphin',
       'Beluga,_White_Whale', 'Bearded_Seal', 'Minke_Whale',
       'Humpback_Whale', 'Southern_Right_Whale', 'White-sided_Dolphin',
       'Narwhal', 'White-beaked_Dolphin', 'Northern_Right_Whale',
       'Frasers_Dolphin', 'Grampus,_Rissos_Dolphin', 'Harp_Seal',
       'Atlantic_Spotted_Dolphin', 'Fin,_Finback_Whale', 'Ross_Seal',
       'Rough-Toothed_Dolphin', 'Killer_Whale',
       'Pantropical_Spotted_Dolphin', 'Short-Finned_Pacific_Pilot_Whale',
       'Bowhead_Whale', 'False_Killer_Whale', 'Melon_Headed_Whale',
       'Long-Finned_Pilot_Whale', 'Striped_Dolphin', 'Leopard_Seal',
       'Walrus', 'Sperm_Whale', 'Common_Dolphin'], dtype=object)

In [ ]:
check_audio_exists("watkins", extensions=["wav"])

In [64]:

def do_process(dsname, cols_to_drop=["Unnamed: 0", "path"]):
    
    train_dfs[dsname]["file_name"] = train_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    train_dfs[dsname]["local_path"] = train_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)
    
    test_dfs[dsname]["file_name"] = test_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    test_dfs[dsname]["local_path"] = test_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)
    
    validation_dfs[dsname]["file_name"] = validation_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    validation_dfs[dsname]["local_path"] = validation_dfs[dsname]["file_name"].apply(lambda x: f"audio/{dsname}/" + x)

    train_dfs[dsname] = train_dfs[dsname].drop(cols_to_drop, axis=1)
    test_dfs[dsname] = test_dfs[dsname].drop(cols_to_drop, axis=1)
    validation_dfs[dsname] = validation_dfs[dsname].drop(cols_to_drop, axis=1)
    

## Humbugdb

In [59]:
train_dfs["humbugdb"].sample(5, replace=False)

,Unnamed: 0,path,label
2325,3814,data/HumBugDB/data/audio/221884.wav,an arabiensis
1794,2948,data/HumBugDB/data/audio/220375.wav,an arabiensis
2379,3891,data/HumBugDB/data/audio/221982.wav,an funestus ss
2948,4862,data/HumBugDB/data/audio/218999.wav,an gambiae ss
4760,7932,data/HumBugDB/data/audio/201732.wav,non-mosquito


In [65]:
do_process("humbugdb")

In [66]:
train_dfs["humbugdb"].sample(5, replace=False)

,label,file_name,local_path
3825,an arabiensis,207637.wav,audio/humbugdb/207637.wav
174,others,330.wav,audio/humbugdb/330.wav
4934,non-mosquito,201274.wav,audio/humbugdb/201274.wav
1126,non-mosquito,1587.wav,audio/humbugdb/1587.wav
4499,non-mosquito,201904.wav,audio/humbugdb/201904.wav


# Egyptian fruit bats

In [68]:
train_dfs["egyptian_fruit_bats"].sample(10)

,Unnamed: 0,path,label
584,954,data/egyptian_fruit_bats/audio/50402.wav,231
24,40,data/egyptian_fruit_bats/audio/87135.wav,215
3922,6547,data/egyptian_fruit_bats/audio/22702.wav,111
4735,7862,data/egyptian_fruit_bats/audio/78067.wav,220
3646,6106,data/egyptian_fruit_bats/audio/63184.wav,211
3847,6431,data/egyptian_fruit_bats/audio/8316.wav,111
1877,3177,data/egyptian_fruit_bats/audio/63978.wav,215
1244,2113,data/egyptian_fruit_bats/audio/86081.wav,216
5257,8722,data/egyptian_fruit_bats/audio/22462.wav,228
2623,4451,data/egyptian_fruit_bats/audio/67639.wav,220


In [69]:
do_process("egyptian_fruit_bats")

In [70]:
train_dfs["egyptian_fruit_bats"].sample(5)

,label,file_name,local_path
4569,210,77437.wav,audio/egyptian_fruit_bats/77437.wav
3551,220,66082.wav,audio/egyptian_fruit_bats/66082.wav
1734,210,76450.wav,audio/egyptian_fruit_bats/76450.wav
1708,216,69181.wav,audio/egyptian_fruit_bats/69181.wav
1647,226,16055.wav,audio/egyptian_fruit_bats/16055.wav


## Dogs

In [71]:
train_dfs["dogs"].sample(10)

,Unnamed: 0,path,label
91,149,data/dogs/wav/Siggy-1-C-1f.wav,Siggy
365,605,data/dogs/wav/Luke-C-2s.wav,Luke
201,330,data/dogs/wav/Roodie-3-C-3c.wav,Roodie
351,587,data/dogs/wav/Luke-C-4b.wav,Luke
297,499,data/dogs/wav/Keri3-A-3a.wav,Keri
46,74,data/dogs/wav/Mac-10-C-8b.wav,Mac
234,389,data/dogs/wav/Roodie-9-C-9c.wav,Roodie
269,448,data/dogs/wav/Farley-7-C-5a.wav,Farley
301,506,data/dogs/wav/Keri-7-C-5g.wav,Keri
346,580,data/dogs/wav/Luke-6-C-6f.wav,Luke


In [72]:
do_process("dogs")

In [73]:
train_dfs["dogs"].sample(5)

,label,file_name,local_path
53,Rudy,Rudy-4-P-4a.wav,audio/dogs/Rudy-4-P-4a.wav
289,Farley,Farley-8-A-7.wav,audio/dogs/Farley-8-A-7.wav
137,Zoe,Zoe-5-P-4c(7_14_00).wav,audio/dogs/Zoe-5-P-4c(7_14_00).wav
328,Freid,Freid-P-8c.wav,audio/dogs/Freid-P-8c.wav
176,Roodie,Roodie-4-P-4d.wav,audio/dogs/Roodie-4-P-4d.wav


## Hiceas

In [74]:
train_dfs["hiceas"].sample(10)

,instruction,path,answer
1401,{hiceas-detection},audio/hiceas-detection/1705_20170903_042852_03...,None
927,{hiceas-detection},audio/hiceas-detection/1705_20170902_010622_37...,None
3045,{hiceas-detection},audio/hiceas-detection/1705_20171002_034935_13...,None
1939,{hiceas-detection},audio/hiceas-detection/1705_20170927_005914_50...,None
1111,{hiceas-detection},audio/hiceas-detection/1705_20170902_191552_03...,None
909,{hiceas-detection},audio/hiceas-detection/1705_20170902_010122_37...,None
929,{hiceas-detection},audio/hiceas-detection/1705_20170902_010622_37...,None
3085,{hiceas-detection},audio/hiceas-detection/1705_20171009_193428_94...,None
1599,{hiceas-detection},audio/hiceas-detection/1705_20170904_213942_22...,None
483,{hiceas-detection},audio/hiceas-detection/1705_20170831_235641_08...,None


In [76]:
train_dfs["hiceas"].answer.unique()

array(['None', "Minke Whale 'boing' call"], dtype=object)

In [77]:
do_process("hiceas", cols_to_drop=["instruction", "path"])

In [79]:
train_dfs["hiceas"].sample(10)

,answer,file_name,local_path
3251,None,1705_20171010_024401_141_006.wav,audio/hiceas/1705_20171010_024401_141_006.wav
417,None,1705_20170831_221156_491_010.wav,audio/hiceas/1705_20170831_221156_491_010.wav
3037,None,1705_20171002_034935_134_001.wav,audio/hiceas/1705_20171002_034935_134_001.wav
356,None,1705_20170831_190456_491_004.wav,audio/hiceas/1705_20170831_190456_491_004.wav
3655,None,1705_20171025_193457_142_003.wav,audio/hiceas/1705_20171025_193457_142_003.wav
596,None,1705_20170901_032341_084_002.wav,audio/hiceas/1705_20170901_032341_084_002.wav
393,None,1705_20170831_212556_491_008.wav,audio/hiceas/1705_20170831_212556_491_008.wav
3607,None,1705_20171025_185657_142_010.wav,audio/hiceas/1705_20171025_185657_142_010.wav
425,None,1705_20170831_222741_084_007.wav,audio/hiceas/1705_20170831_222741_084_007.wav
2881,None,1705_20171001_235545_475_010.wav,audio/hiceas/1705_20171001_235545_475_010.wav


## Dcase

In [81]:
train_dfs["dcase"].sample(5)

,instruction,path,answer
40224,{dcase-detection},audio/dcase-detection/BUK5_20161101_002104a.01...,None
30461,{dcase-detection},audio/dcase-detection/y1.032_017.wav,None
20203,{dcase-detection},audio/dcase-detection/2015-09-25_04-00-00_unit...,None
36844,{dcase-detection},audio/dcase-detection/BUK1_20181011_001004.010...,None
26606,{dcase-detection},audio/dcase-detection/e1.022_056.wav,None


In [82]:
train_dfs["dcase"].answer.unique()

array(['None', 'American Redstart', 'Other Animal',
       'Rose-breasted Grosbeak', 'Chipping Sparrow',
       'Chipping Sparrow, American Redstart', 'Ovenbird',
       'Meerkat short note', 'American Redstart, Other Animal',
       'Meerkat short note, Common Yellowthroat',
       'Meerkat short note, Common Yellowthroat, Other Animal',
       'Meerkat short note, Other Animal', "Swainson's Thrush",
       'Common Yellowthroat, Other Animal', 'Common Yellowthroat',
       'Rose-breasted Grosbeak, Other Animal',
       'Common Yellowthroat, American Redstart',
       "Swainson's Thrush, Common Yellowthroat, Other Animal",
       "American Redstart, Swainson's Thrush", 'Bay-breasted Warbler',
       'American Redstart, Bay-breasted Warbler',
       'Common Yellowthroat, Rose-breasted Grosbeak',
       'Bay-breasted Warbler, Rose-breasted Grosbeak',
       'Meerkat close call', 'Savannah Sparrow',
       'Meerkat close call, Bay-breasted Warbler', 'Hyena squitter',
       'Hyena groan a

In [83]:
do_process("dcase", ["instruction", "path"])

In [84]:
train_dfs["dcase"].sample(5)

,answer,file_name,local_path
28304,None,h1.023_043.wav,audio/dcase/h1.023_043.wav
1010,None,2015-09-11_06-00-00_unit07.002_007.wav,audio/dcase/2015-09-11_06-00-00_unit07.002_007...
25088,None,2015-10-14_23-59-59_unit05.061_013.wav,audio/dcase/2015-10-14_23-59-59_unit05.061_013...
11809,None,2015-09-04_08-04-59_unit03.064_009.wav,audio/dcase/2015-09-04_08-04-59_unit03.064_009...
34747,None,a1.034_055.wav,audio/dcase/a1.034_055.wav


## Enabirds

In [85]:
train_dfs["enabirds"].sample(5)

,instruction,path,answer
512,{enabirds-detection},audio/enabirds-detection/Recording_1_Segment_0...,None
1098,{enabirds-detection},audio/enabirds-detection/Recording_1_Segment_1...,"Eastern Towhee, Hermit Thrush"
9340,{enabirds-detection},audio/enabirds-detection/Recording_2_Segment_0...,"Wood Thrush, American Redstart, Black-capped C..."
4214,{enabirds-detection},audio/enabirds-detection/Recording_4_Segment_2...,Eastern Towhee
10969,{enabirds-detection},audio/enabirds-detection/Recording_4_Segment_0...,Eastern Towhee


In [87]:
do_process("enabirds", ["instruction", "path"])

In [88]:
train_dfs["enabirds"].sample(5)

,answer,file_name,local_path
2929,"Wood Thrush, American Redstart, American Crow,...",Recording_2_Segment_14.000_038.wav,audio/enabirds/Recording_2_Segment_14.000_038.wav
2524,"Red-eyed Vireo, American Redstart, Wood Thrush",Recording_2_Segment_07.000_046.wav,audio/enabirds/Recording_2_Segment_07.000_046.wav
13320,"Eastern Towhee, Blue-gray Gnatcatcher",Recording_4_Segment_24.001_045.wav,audio/enabirds/Recording_4_Segment_24.001_045.wav
3072,Eastern Towhee,Recording_4_Segment_02.000_004.wav,audio/enabirds/Recording_4_Segment_02.000_004.wav
8528,None,Recording_1_Segment_34.002_032.wav,audio/enabirds/Recording_1_Segment_34.002_032.wav


## Rfcx

In [89]:
train_dfs["rfcx"].sample(5)

,instruction,path,answer
11622,{rfcx-detection},audio/rfcx-detection/5f9b4785b_006.wav,None
4145,{rfcx-detection},audio/rfcx-detection/206f79102_009.wav,None
1150,{rfcx-detection},audio/rfcx-detection/09315d9bf_006.wav,None
5188,{rfcx-detection},audio/rfcx-detection/2a1924793_007.wav,None
23315,{rfcx-detection},audio/rfcx-detection/c12e0a62b_006.wav,None


In [90]:
do_process("rfcx", ["instruction", "path"])

In [91]:
train_dfs["rfcx"].sample(5)

,answer,file_name,local_path
15002,Dwarf coqui,7b287d190_009.wav,audio/rfcx/7b287d190_009.wav
12384,None,666dd7d48_009.wav,audio/rfcx/666dd7d48_009.wav
23544,None,c350857cd_004.wav,audio/rfcx/c350857cd_004.wav
17950,None,94d96ee35_009.wav,audio/rfcx/94d96ee35_009.wav
26881,None,dca449f04_008.wav,audio/rfcx/dca449f04_008.wav


## Hainan gibbons

In [93]:
train_dfs["hainan_gibbons"].sample(5)

,instruction,path,answer
4073,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
12812,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
2768,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3AC_0+1_201...,None
19735,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3BD_0+1_201...,None
21262,{hainan-gibbons-detection},audio/hainan-gibbons-detection/HGSM3BD_0+1_201...,None


In [94]:
do_process("hainan_gibbons", ["instruction", "path"])

In [95]:
train_dfs["hainan_gibbons"].sample(5)

,answer,file_name,local_path
5961,None,HGSM3AC_0+1_20160312_055400.135_016.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160312_0554...
14669,None,HGSM3A_0+1_20160304_060000.075_024.wav,audio/hainan_gibbons/HGSM3A_0+1_20160304_06000...
17042,None,HGSM3A_0+1_20160304_060000.321_019.wav,audio/hainan_gibbons/HGSM3A_0+1_20160304_06000...
25664,None,HGSM3BD_0+1_20160401_053700.252_028.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160401_0537...
26856,None,HGSM3BD_0+1_20160401_053700.378_002.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160401_0537...


## esc50-all

In [96]:
train_dfs["esc50-all"].sample(5)

,instruction,path,label
715,{esc50-all},audio/esc50/2-74361-A-47.wav,airplane
251,{esc50-all},audio/esc50/1-51436-A-17.wav,pouring_water
454,{esc50-all},audio/esc50/2-108767-B-9.wav,crow
1076,{esc50-all},audio/esc50/3-164120-A-11.wav,sea_waves
1139,{esc50-all},audio/esc50/3-203374-A-39.wav,glass_breaking


In [ ]:
do_process("esc50-all", ["instruction", "path"])

In [99]:
train_dfs["esc50-all"].sample(5)

,label,file_name,local_path
506,sheep,2-119161-A-8.wav,audio/esc50-all/2-119161-A-8.wav
781,brushing_teeth,2-94230-A-27.wav,audio/esc50-all/2-94230-A-27.wav
1165,siren,3-51909-B-42.wav,audio/esc50-all/3-51909-B-42.wav
768,coughing,2-87412-A-24.wav,audio/esc50-all/2-87412-A-24.wav
924,train,3-136451-A-45.wav,audio/esc50-all/3-136451-A-45.wav


## Speech-commands

In [101]:
train_dfs["speech-commands"].sample(5)

,instruction,path,label
20509,{speech-commands-classification},audio/speech_commands/four/2e75d37a_nohash_4.wav,four
58753,{speech-commands-classification},audio/speech_commands/six/023a61ad_nohash_0.wav,six
69026,{speech-commands-classification},audio/speech_commands/tree/d3a18257_nohash_0.wav,tree
60338,{speech-commands-classification},audio/speech_commands/six/89947bd7_nohash_2.wav,six
438,{speech-commands-classification},audio/speech_commands/backward/57152045_nohash...,backward


In [102]:
train_dfs["speech-commands"]["path"].apply(lambda x: "wav" in x).sum()

np.int64(84843)

In [103]:
train_dfs["speech-commands"].shape

(84843, 3)

In [104]:
def do_process_speech_commands(dsname, cols_to_drop=["Unnamed: 0", "path"]):
    
    train_dfs[dsname]["file_name"] = train_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    train_dfs[dsname]["local_path"] = train_dfs[dsname]["path"].copy()
    
    test_dfs[dsname]["file_name"] = test_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    test_dfs[dsname]["local_path"] = test_dfs[dsname]["path"].copy()
    
    validation_dfs[dsname]["file_name"] = validation_dfs[dsname]["path"].apply(lambda x: Path(x).name)
    validation_dfs[dsname]["local_path"] = validation_dfs[dsname]["path"].copy()

    train_dfs[dsname] = train_dfs[dsname].drop(cols_to_drop, axis=1)
    test_dfs[dsname] = test_dfs[dsname].drop(cols_to_drop, axis=1)
    validation_dfs[dsname] = validation_dfs[dsname].drop(cols_to_drop, axis=1)

In [105]:
do_process_speech_commands("speech-commands", ["instruction", "path"])

In [106]:
train_dfs["speech-commands"].sample(5)

,label,file_name,local_path
78602,yes,13199b79_nohash_0.wav,audio/speech_commands/yes/13199b79_nohash_0.wav
80444,yes,a84dee7b_nohash_1.wav,audio/speech_commands/yes/a84dee7b_nohash_1.wav
49484,one,91ffb786_nohash_1.wav,audio/speech_commands/one/91ffb786_nohash_1.wav
3974,bird,a19452d5_nohash_1.wav,audio/speech_commands/bird/a19452d5_nohash_1.wav
30734,left,0717b9f6_nohash_1.wav,audio/speech_commands/left/0717b9f6_nohash_1.wav


# Save all as individual jsonls

In [107]:
train_dfs.keys()

dict_keys(['cbi', 'watkins', 'humbugdb', 'egyptian_fruit_bats', 'dogs', 'hiceas', 'dcase', 'enabirds', 'rfcx', 'hainan_gibbons', 'esc50-all', 'speech-commands'])

In [108]:
beans_root

GSPath('gs://esp-ml-datasets/beans/v0.1.0/raw/')

In [ ]:
train_dfs["cbi"].to_json(str(beans_root / "cbi_train.jsonl"), lines=True, orient="records")
train_dfs["watkins"].to_json(str(beans_root / "watkins_train.jsonl"), lines=True, orient="records")
train_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_train.jsonl"), lines=True, orient="records")
train_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_train.jsonl"), lines=True, orient="records")
train_dfs["dogs"].to_json(str(beans_root / "dogs_train.jsonl"), lines=True, orient="records")
train_dfs["hiceas"].to_json(str(beans_root / "hiceas_train.jsonl"), lines=True, orient="records")
train_dfs["dcase"].to_json(str(beans_root / "dcase_train.jsonl"), lines=True, orient="records")
train_dfs["enabirds"].to_json(str(beans_root / "enabirds_train.jsonl"), lines=True, orient="records")
train_dfs["rfcx"].to_json(str(beans_root / "rfcx_train.jsonl"), lines=True, orient="records")
train_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_train.jsonl"), lines=True, orient="records")
train_dfs["esc50-all"].to_json(str(beans_root / "esc50_train.jsonl"), lines=True, orient="records")
train_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_train.jsonl"), lines=True, orient="records")

In [122]:
test_dfs["cbi"].to_json(str(beans_root / "cbi_test.jsonl"), lines=True, orient="records")
test_dfs["watkins"].to_json(str(beans_root / "watkins_test.jsonl"), lines=True, orient="records")
test_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_test.jsonl"), lines=True, orient="records")
test_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_test.jsonl"), lines=True, orient="records")
test_dfs["dogs"].to_json(str(beans_root / "dogs_test.jsonl"), lines=True, orient="records")
test_dfs["hiceas"].to_json(str(beans_root / "hiceas_test.jsonl"), lines=True, orient="records")
test_dfs["dcase"].to_json(str(beans_root / "dcase_test.jsonl"), lines=True, orient="records")
test_dfs["enabirds"].to_json(str(beans_root / "enabirds_test.jsonl"), lines=True, orient="records")
test_dfs["rfcx"].to_json(str(beans_root / "rfcx_test.jsonl"), lines=True, orient="records")
test_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_test.jsonl"), lines=True, orient="records")
test_dfs["esc50-all"].to_json(str(beans_root / "esc50_test.jsonl"), lines=True, orient="records")
test_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_test.jsonl"), lines=True, orient="records")

In [123]:
validation_dfs["cbi"].to_json(str(beans_root / "cbi_val.jsonl"), lines=True, orient="records")
validation_dfs["watkins"].to_json(str(beans_root / "watkins_val.jsonl"), lines=True, orient="records")
validation_dfs["humbugdb"].to_json(str(beans_root / "humbugdb_val.jsonl"), lines=True, orient="records")
validation_dfs["egyptian_fruit_bats"].to_json(str(beans_root / "egyptian_fruit_bats_val.jsonl"), lines=True, orient="records")
validation_dfs["dogs"].to_json(str(beans_root / "dogs_val.jsonl"), lines=True, orient="records")
validation_dfs["hiceas"].to_json(str(beans_root / "hiceas_val.jsonl"), lines=True, orient="records")
validation_dfs["dcase"].to_json(str(beans_root / "dcase_val.jsonl"), lines=True, orient="records")
validation_dfs["enabirds"].to_json(str(beans_root / "enabirds_val.jsonl"), lines=True, orient="records")
validation_dfs["rfcx"].to_json(str(beans_root / "rfcx_val.jsonl"), lines=True, orient="records")
validation_dfs["hainan_gibbons"].to_json(str(beans_root / "hainan_gibbons_val.jsonl"), lines=True, orient="records")
validation_dfs["esc50-all"].to_json(str(beans_root / "esc50_val.jsonl"), lines=True, orient="records")
validation_dfs["speech-commands"].to_json(str(beans_root / "speech_commands_val.jsonl"), lines=True, orient="records")

# Combine all train, test and val

In [124]:
full_train_df = []

for k, df in train_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_train_df.append(df)

In [125]:
full_train_df = pd.concat(full_train_df)
full_train_df.shape

(231819, 4)

In [148]:
full_train_df.sample(10)

,label,file_name,local_path,source_dataset
29474,learn,195c120a_nohash_2.wav,audio/speech_commands/learn/195c120a_nohash_2.wav,speech_commands
27162,None,HGSM3BD_0+1_20160401_053700.408_018.wav,audio/hainan_gibbons/HGSM3BD_0+1_20160401_0537...,hainan_gibbons
68019,tree,18c54a68_nohash_0.wav,audio/speech_commands/tree/18c54a68_nohash_0.wav,speech_commands
39260,no,365908bd_nohash_0.wav,audio/speech_commands/no/365908bd_nohash_0.wav,speech_commands
53294,right,cb802c63_nohash_3.wav,audio/speech_commands/right/cb802c63_nohash_3.wav,speech_commands
5394,None,y1.000_025.wav,audio/dcase/y1.000_025.wav,dcase
8984,None,2015-09-04_08-04-59_unit03.016_016.wav,audio/dcase/2015-09-04_08-04-59_unit03.016_016...,dcase
8677,None,HGSM3AC_0+1_20160312_055400.417_006.wav,audio/hainan_gibbons/HGSM3AC_0+1_20160312_0554...,hainan_gibbons
3924,"Eastern Towhee, Black-capped Chickadee",Recording_4_Segment_16.000_030.wav,audio/enabirds/Recording_4_Segment_16.000_030.wav,enabirds
1494,None,2015-09-11_06-00-00_unit07.010_019.wav,audio/dcase/2015-09-11_06-00-00_unit07.010_019...,dcase


In [129]:
full_train_df["label"].dtype

string[python]

In [128]:
full_train_df["label"] = full_train_df["label"].astype("string")

In [130]:
full_train_df = full_train_df.astype("string")

In [131]:
full_train_df.to_csv("beans_train.csv", index=False)

In [136]:
full_train_df.to_csv(str(beans_root / "beans_train.csv"), index=False)

In [134]:
dd = pd.read_csv("beans_train.csv", keep_default_na=False, na_values=[''])

In [135]:
dd["label"].isnull().sum()

np.int64(0)

In [137]:
full_test_df = []

for k, df in test_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_test_df.append(df)

full_test_df = pd.concat(full_test_df).astype("string")

In [138]:
full_test_df.shape

(68044, 4)

In [141]:
full_test_df.to_csv(str(beans_root / "beans_test.csv"), index=False)

In [142]:
full_val_df = []

for k, df in validation_dfs.items():

    if "answer" in list(df.columns):
        df = df.rename(columns={"answer": "label"})

    if k == "esc50-all":
        dsname = "esc50"
    elif k == "speech-commands":
        dsname = "speech_commands"
    else:
        dsname = k

    df["source_dataset"] = [dsname] * len(df)
    
    df = df.reset_index(drop=True)
    full_val_df.append(df)

full_val_df = pd.concat(full_val_df).astype("string")

In [143]:
full_val_df.to_csv(str(beans_root / "beans_val.csv"), index=False)